In [1]:
# Load env variables
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from anthropic import Anthropic

client=Anthropic()
model="claude-opus-5"

In [3]:
def add_user_message(messages, text):
    user_message={"role":"user", "content":text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message={"role":"assistant", "content":text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=None):
    params={
        "model":model,
        "max_tokens":1000,
        "messages":messages
    }

    #We only add system and temperature as parameters if they are defined
    if system is not None:
        params["system"]=system

    if temperature is not None:
            params["temperature"]=temperature

    message=client.messages.create(
        **params
    )
    print(message)
    for block in message.content:
        if block.type=='text':
            return block.text
        

In [7]:
messages=[]

add_user_message(messages, "Write a 1 sentence description of Claude")

stream=client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages, 
    stream=True
)

#This sends back several types of events, ContentBlockDelta is the chunks of the actual text
for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_011CeSS7KAxqL6vubTteLSFk', container=None, content=[], model='claude-opus-5', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=20, output_tokens=7, output_tokens_details=None, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=ThinkingBlock(signature='', thinking='', type='thinking'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=ThinkingDelta(thinking='', type='thinking_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=SignatureDelta(signature='CAISvwIKjgEIERgCKkCoDcOpCRnwTvRE9yhiKibDjMoOk7f2ncJOvIVGN0HKToN+6pbZmP+LMAAASTDF0lha8lR8x9IWgKzoXhd5it6dMg1jbGF1ZGUtb3B1cy01OAFCCHRoaW5raW5nWiQzMjFiNDg2ZC02O

In [8]:
#This uses the SDK's simplified streaming interface - we access stream.text_stream
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")

Claude is an AI assistant made by Anthropic, designed to be helpful, harmless, and honest while assisting with tasks like writing, analysis, coding, and open-ended conversation.

In [ ]:
#Get the complete message for database storage
final_message = stream.get_final_message()